# NATCARB 2022 Comparison and Standardization

## Objective

This notebook evaluates how a postdoc-modified (Dr. Emily Nishikawa) 2022 NATCARB archive differs from the original online NATCARB dataset previously explored.

The goal is to determine whether the 2022 version should be interpreted as:

- a direct copy of the original NATCARB release with additional attributes,
- a corrected or updated version of existing NATCARB records,
- a derived dataset that restructures or reinterprets the original source, or
- a combination of these.

This comparison will be completed before developing the standardized NATCARB GeoPackage used in the broader Canadian geological CO₂ storage database.

## Questions

The notebook will investigate:

1. How do the archive contents differ between the original online NATCARB source and the 2022 postdoc archive?
2. Are the same spatial layers present in both datasets?
3. Have layer schemas changed?
4. Which fields were retained, removed, added, or renamed?
5. Are existing NATCARB attribute values preserved or modified?
6. Were geometries changed?
7. Were records added, removed, split, merged, or otherwise restructured?
8. What do any new identifiers introduced in the 2022 version represent?
9. Do the 2022 modifications improve geological linkage between storage-resource records, grid cells, formations, or basins?
10. How should the original and modified datasets be represented in a standardized NATCARB GeoPackage while preserving source provenance?

## Interpretation approach

The original online NATCARB release will remain the reference source dataset.

The 2022 archive will initially be treated as a separate derived or augmented source rather than assumed to supersede the original release.

Changes identified during comparison will be classified where possible as:

- unchanged source information,
- metadata enrichment,
- geological identity enrichment,
- capacity modification,
- reservoir-property modification,
- geometry modification,
- newly introduced records,
- removed records,
- structural transformation, or
- uncertain modification.

This distinction is important because additions made during later research workflows should not be silently attributed to the original NATCARB source.

## Intended output

The final result of this notebook should provide enough information to define:

- the canonical NATCARB source fields that should be preserved,
- the provenance of fields introduced in 2022,
- the appropriate record identifiers and relationships,
- whether multiple NATCARB releases or derived versions should be retained,
- how storage-resource observations should relate to geological formations or basins,
- how capacity and reservoir properties should be represented,
- and how NATCARB should ultimately be standardized into a GeoPackage suitable for integration with Canadian geological storage datasets.

The standardized product should preserve both the original source information and any later modifications without obscuring their lineage.

In [14]:
# ---------------------------------------------------------------------------
# Source paths
# ---------------------------------------------------------------------------

import zipfile
from pathlib import Path

import geopandas as gpd
import hashlib
import numpy as np
import pandas as pd
import pyogrio


# ---------------------------------------------------------------------------
# Original online NATCARB source
# ---------------------------------------------------------------------------

# Previously downloaded and extracted NATCARB dataset
NATCARB_OFFICIAL_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage\NATCARB"
)


# ---------------------------------------------------------------------------
# Postdoc NATCARB source files
# ---------------------------------------------------------------------------

POSTDOC_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage"
    r"\Feras Canada CO2 Storage"
    r"\Geospatial Data"
    r"\NATCARB Atlas Saline Basin 10km Grid"
)

# April 2022 updated 10 km saline grid
POSTDOC_2022_GRID_ZIP = (
    POSTDOC_DIR
    / "natcarb_saline_10km_grid_shapefile_3_31_2022.zip"
)

# Formation / reservoir polygon layer used by the 2022 update
POSTDOC_SALINE_POLY_ZIP = (
    POSTDOC_DIR
    / "natcarb_saline_poly_shapefile.zip"
)

# 2015 NATCARB saline 10 km grid retained with the postdoc files
POSTDOC_2015_GRID_ZIP = (
    POSTDOC_DIR
    / "saline10k_1502.zip"
)


# ---------------------------------------------------------------------------
# Validate configured sources
# ---------------------------------------------------------------------------

sources = {
    "Official extracted NATCARB": NATCARB_OFFICIAL_DIR,
    "Postdoc 2022 saline grid": POSTDOC_2022_GRID_ZIP,
    "Postdoc saline polygons": POSTDOC_SALINE_POLY_ZIP,
    "Postdoc 2015 saline grid": POSTDOC_2015_GRID_ZIP,
}

print("Configured NATCARB sources:\n")

for label, path in sources.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:7} | {label}")
    print(f"          {path}")

Configured NATCARB sources:

FOUND   | Official extracted NATCARB
          C:\Users\aviga\Research\potential data\Storage\NATCARB
FOUND   | Postdoc 2022 saline grid
          C:\Users\aviga\Research\potential data\Storage\Feras Canada CO2 Storage\Geospatial Data\NATCARB Atlas Saline Basin 10km Grid\natcarb_saline_10km_grid_shapefile_3_31_2022.zip
FOUND   | Postdoc saline polygons
          C:\Users\aviga\Research\potential data\Storage\Feras Canada CO2 Storage\Geospatial Data\NATCARB Atlas Saline Basin 10km Grid\natcarb_saline_poly_shapefile.zip
FOUND   | Postdoc 2015 saline grid
          C:\Users\aviga\Research\potential data\Storage\Feras Canada CO2 Storage\Geospatial Data\NATCARB Atlas Saline Basin 10km Grid\saline10k_1502.zip


In [3]:
# ---------------------------------------------------------------------------
# Inventory source contents
# ---------------------------------------------------------------------------

def inventory_zip(zip_path: Path) -> pd.DataFrame:
    """Return a file inventory for a ZIP archive."""
    with zipfile.ZipFile(zip_path, "r") as archive:
        records = [
            {
                "source": zip_path.name,
                "path": info.filename,
                "filename": Path(info.filename).name,
                "suffix": Path(info.filename).suffix.lower(),
                "size_bytes": info.file_size,
                "compressed_bytes": info.compress_size,
                "is_directory": info.is_dir(),
            }
            for info in archive.infolist()
        ]

    return pd.DataFrame(records)


def inventory_directory(directory: Path) -> pd.DataFrame:
    """Return a recursive file inventory for an extracted directory."""
    records = [
        {
            "source": directory.name,
            "path": str(path.relative_to(directory)),
            "filename": path.name,
            "suffix": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "compressed_bytes": None,
            "is_directory": False,
        }
        for path in directory.rglob("*")
        if path.is_file()
    ]

    return pd.DataFrame(records)


# ---------------------------------------------------------------------------
# Build inventories
# ---------------------------------------------------------------------------

official_inventory = inventory_directory(
    NATCARB_OFFICIAL_DIR
)

postdoc_2022_inventory = inventory_zip(
    POSTDOC_2022_GRID_ZIP
)

postdoc_poly_inventory = inventory_zip(
    POSTDOC_SALINE_POLY_ZIP
)

postdoc_2015_inventory = inventory_zip(
    POSTDOC_2015_GRID_ZIP
)


# ---------------------------------------------------------------------------
# Combine and summarize
# ---------------------------------------------------------------------------

inventory = pd.concat(
    [
        official_inventory,
        postdoc_2022_inventory,
        postdoc_poly_inventory,
        postdoc_2015_inventory,
    ],
    ignore_index=True,
)

summary = (
    inventory
    .groupby("source", dropna=False)
    .agg(
        files=("filename", "count"),
        total_size_mb=("size_bytes", lambda x: x.sum() / 1e6),
        unique_suffixes=("suffix", lambda x: sorted(set(x))),
    )
    .reset_index()
)

display(summary)

print("\nDetailed inventory:")
display(
    inventory[
        [
            "source",
            "path",
            "suffix",
            "size_bytes",
        ]
    ].sort_values(
        ["source", "path"]
    )
)

,source,files,total_size_mb,unique_suffixes
0,NATCARB,131,361.101090,"[, .atx, .csv, .freelist, .gdbindexes, .gdbtab..."
1,natcarb_saline_10km_grid_shapefile_3_31_2022.zip,9,1476.394275,"[, .cpg, .dbf, .prj, .sbn, .sbx, .shp, .shx, ...."
2,natcarb_saline_poly_shapefile.zip,9,4.453555,"[, .cpg, .dbf, .prj, .sbn, .sbx, .shp, .shx, ...."
3,saline10k_1502.zip,8,98.962721,"[.cpg, .dbf, .prj, .sbn, .sbx, .shp, .shx, .xml]"



Detailed inventory:


,source,path,suffix,size_bytes
1,NATCARB,Diagnostics\assessment_volume_check.csv,.csv,154
2,NATCARB,Diagnostics\duplicate_variation.csv,.csv,2340
3,NATCARB,Diagnostics\qa_summary.csv,.csv,96
4,NATCARB,Diagnostics\saline_10k.csv,.csv,28386488
5,NATCARB,Metadata_v1502\FGDC_Plus_DASC2.xsl,.xsl,237561
...,...,...,...,...
152,saline10k_1502.zip,saline10k_1502.sbn,.sbn,1785876
153,saline10k_1502.zip,saline10k_1502.sbx,.sbx,50100
154,saline10k_1502.zip,saline10k_1502.shp,.shp,25387900
155,saline10k_1502.zip,saline10k_1502.shp.xml,.xml,54388


In [5]:
# ---------------------------------------------------------------------------
# Locate saline-related files in the official extracted NATCARB directory
# ---------------------------------------------------------------------------

saline_matches = sorted(
    path
    for path in NATCARB_OFFICIAL_DIR.rglob("*")
    if path.is_file()
    and "saline" in path.name.lower()
)

print(
    f"Found {len(saline_matches):,} files containing "
    "'saline' in the filename.\n"
)

for path in saline_matches:
    print(path.relative_to(NATCARB_OFFICIAL_DIR))

Found 5 files containing 'saline' in the filename.

Diagnostics\saline_10k.csv
Metadata_v1502\NATCARB_Saline_10K_v1502.pdf
Metadata_v1502\NATCARB_Saline_10K_v1502.xml
Metadata_v1502\NATCARB_Saline_Poly_v1502.pdf
Metadata_v1502\NATCARB_Saline_Poly_v1502.xml


In [7]:
# ---------------------------------------------------------------------------
# Find geodatabases in the official NATCARB directory
# ---------------------------------------------------------------------------

gdb_paths = sorted(
    path
    for path in NATCARB_OFFICIAL_DIR.rglob("*.gdb")
    if path.is_dir()
)

print(f"Found {len(gdb_paths)} geodatabase(s):\n")

for gdb_path in gdb_paths:
    print(gdb_path.relative_to(NATCARB_OFFICIAL_DIR))


# ---------------------------------------------------------------------------
# List layers in each geodatabase
# ---------------------------------------------------------------------------

gdb_layer_records = []

for gdb_path in gdb_paths:
    layers = pyogrio.list_layers(gdb_path)

    print(f"\n{gdb_path.name}")
    print("-" * len(gdb_path.name))

    for layer_name, geometry_type in layers:
        print(
            f"{layer_name:50} "
            f"{str(geometry_type):20}"
        )

        gdb_layer_records.append(
            {
                "gdb": gdb_path.name,
                "gdb_path": str(gdb_path),
                "layer": layer_name,
                "geometry_type": geometry_type,
            }
        )

gdb_layers = pd.DataFrame(gdb_layer_records)

display(gdb_layers)

Found 1 geodatabase(s):

NATCARB_v1502.gdb

NATCARB_v1502.gdb
-----------------
Domain_State                                       None                
Domain_Fuel                                        None                
Domain_Overlap                                     None                
Domain_Duplicate                                   None                
Domain_ARRA                                        None                
Domain_Source_Types                                None                
Domain_Med_Calced                                  None                
NATCARB_Coal_10K_v1502                             MultiPolygon        
Domain_Partnership                                 None                
NATCARB_Coal_Poly_v1502                            MultiPolygon        
Domain_Oil_Gas                                     None                
Domain_Assessed                                    None                
NATCARB_Saline_Poly_v1502                          Multi

,gdb,gdb_path,layer,geometry_type
0,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_State,NaN
1,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Fuel,NaN
2,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Overlap,NaN
3,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Duplicate,NaN
4,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_ARRA,NaN
5,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Source_Types,NaN
6,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Med_Calced,NaN
7,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,NATCARB_Coal_10K_v1502,MultiPolygon
8,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,Domain_Partnership,NaN
9,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,NATCARB_Coal_Poly_v1502,MultiPolygon


In [8]:
# ---------------------------------------------------------------------------
# Identify candidate saline layers
# ---------------------------------------------------------------------------

saline_layers = gdb_layers[
    gdb_layers["layer"]
    .str.contains(
        r"saline|10k|10km",
        case=False,
        regex=True,
        na=False,
    )
].copy()

display(saline_layers)

,gdb,gdb_path,layer,geometry_type
7,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,NATCARB_Coal_10K_v1502,MultiPolygon
12,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,NATCARB_Saline_Poly_v1502,MultiPolygon
13,NATCARB_v1502.gdb,C:\Users\aviga\Research\potential data\Storage...,NATCARB_Saline_10K_v1502,MultiPolygon


In [9]:
# ---------------------------------------------------------------------------
# Load official v1502 and postdoc 2015 saline 10 km layers
# ---------------------------------------------------------------------------

import tempfile
import zipfile

import geopandas as gpd
import pandas as pd


# ---------------------------------------------------------------------------
# Official NATCARB v1502 saline 10 km layer
# ---------------------------------------------------------------------------

OFFICIAL_GDB = gdb_paths[0]
OFFICIAL_SALINE_10K_LAYER = "NATCARB_Saline_10K_v1502"

official_saline_10k = gpd.read_file(
    OFFICIAL_GDB,
    layer=OFFICIAL_SALINE_10K_LAYER,
)

print("Official v1502 saline 10 km layer")
print("------------------------------------")
print(f"Rows:          {len(official_saline_10k):,}")
print(f"Columns:       {len(official_saline_10k.columns):,}")
print(f"CRS:           {official_saline_10k.crs}")
print(f"Geometry type: {official_saline_10k.geom_type.value_counts().to_dict()}")


# ---------------------------------------------------------------------------
# Postdoc copy of the 2015 saline 10 km shapefile
# ---------------------------------------------------------------------------

with tempfile.TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    with zipfile.ZipFile(
        POSTDOC_2015_GRID_ZIP,
        "r",
    ) as archive:
        archive.extractall(temp_dir)

    shapefiles = sorted(
        temp_dir.rglob("*.shp")
    )

    if len(shapefiles) != 1:
        raise ValueError(
            "Expected exactly one shapefile in "
            f"{POSTDOC_2015_GRID_ZIP.name}, "
            f"found {len(shapefiles)}: "
            f"{[path.name for path in shapefiles]}"
        )

    postdoc_2015_saline_10k = gpd.read_file(
        shapefiles[0]
    )


print("\nPostdoc 2015 saline 10 km layer")
print("--------------------------------")
print(f"Rows:          {len(postdoc_2015_saline_10k):,}")
print(f"Columns:       {len(postdoc_2015_saline_10k.columns):,}")
print(f"CRS:           {postdoc_2015_saline_10k.crs}")
print(
    "Geometry type:",
    postdoc_2015_saline_10k
    .geom_type
    .value_counts()
    .to_dict(),
)


# ---------------------------------------------------------------------------
# Basic structural comparison
# ---------------------------------------------------------------------------

structural_comparison = pd.DataFrame(
    [
        {
            "dataset": "Official NATCARB v1502 GDB",
            "rows": len(official_saline_10k),
            "columns": len(official_saline_10k.columns),
            "crs": str(official_saline_10k.crs),
        },
        {
            "dataset": "Postdoc saline10k_1502 ZIP",
            "rows": len(postdoc_2015_saline_10k),
            "columns": len(postdoc_2015_saline_10k.columns),
            "crs": str(postdoc_2015_saline_10k.crs),
        },
    ]
)

display(structural_comparison)

Official v1502 saline 10 km layer
------------------------------------
Rows:          186,675
Columns:       24
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",45],PARAMETER["longitude_of_center",-100],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry type: {'MultiPolygon': 186675}

Postdoc 2015 saline 10 km layer
--------------------------------
Rows:          186,675
Columns:       24
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION

,dataset,rows,columns,crs
0,Official NATCARB v1502 GDB,186675,24,"PROJCS[""Lambert Azimuthal Equal-area"",GEOGCS[""..."
1,Postdoc saline10k_1502 ZIP,186675,24,"PROJCS[""Lambert Azimuthal Equal-area"",GEOGCS[""..."


In [10]:
# ---------------------------------------------------------------------------
# Compare v1502 schemas
# ---------------------------------------------------------------------------

official_schema = pd.DataFrame(
    {
        "field": official_saline_10k.columns,
        "official_dtype": [
            str(official_saline_10k[column].dtype)
            for column in official_saline_10k.columns
        ],
    }
)

postdoc_schema = pd.DataFrame(
    {
        "field": postdoc_2015_saline_10k.columns,
        "postdoc_dtype": [
            str(postdoc_2015_saline_10k[column].dtype)
            for column in postdoc_2015_saline_10k.columns
        ],
    }
)


# ---------------------------------------------------------------------------
# Compare field presence
# ---------------------------------------------------------------------------

schema_comparison = (
    official_schema
    .merge(
        postdoc_schema,
        on="field",
        how="outer",
        indicator=True,
    )
    .rename(
        columns={
            "_merge": "field_status",
        }
    )
)

schema_comparison["field_status"] = (
    schema_comparison["field_status"]
    .map(
        {
            "both": "shared",
            "left_only": "official_only",
            "right_only": "postdoc_only",
        }
    )
)

schema_comparison["same_dtype"] = (
    schema_comparison["official_dtype"]
    == schema_comparison["postdoc_dtype"]
)

display(schema_comparison)


# ---------------------------------------------------------------------------
# Compare field order
# ---------------------------------------------------------------------------

official_columns = list(
    official_saline_10k.columns
)

postdoc_columns = list(
    postdoc_2015_saline_10k.columns
)

print(
    "Same field names:",
    set(official_columns) == set(postdoc_columns),
)

print(
    "Same field order:",
    official_columns == postdoc_columns,
)

print(
    "Same pandas dtypes:",
    schema_comparison["same_dtype"].all(),
)

,field,official_dtype,postdoc_dtype,field_status,same_dtype
0,ARRA_PROJE,NaN,str,postdoc_only,False
1,ARRA_PROJECT,str,NaN,official_only,False
2,ASSESSED,int16,int32,shared,False
3,BASIN_NAME,str,str,shared,True
4,COL_ROW,str,str,shared,True
5,CYCLE_OF_L,NaN,str,postdoc_only,False
6,CYCLE_OF_LAST_UPDATE,str,NaN,official_only,False
7,DEPTH_FT,float64,int64,shared,False
8,DUPLICATE,int16,int32,shared,False
9,MED_CALCED,int16,int32,shared,False


Same field names: False
Same field order: False
Same pandas dtypes: False


In [11]:
# ---------------------------------------------------------------------------
# Harmonize v1502 field names across FileGDB and shapefile representations
# ---------------------------------------------------------------------------

FIELD_CROSSWALK = {
    "PARTNERSHI": "PARTNERSHIP",
    "ARRA_PROJE": "ARRA_PROJECT",
    "RESOURCE_N": "RESOURCE_NAME",
    "RSC_AREA_C": "RSC_AREA_CELL",
    "SALINITY_T": "SALINITY_TDS",
    "PRESSURE_P": "PRESSURE_PSI",
    "TEMPERATUR": "TEMPERATURE_F",
    "POROSITY_P": "POROSITY_PCT",
    "PERMEABILI": "PERMEABILITY_mD",
    "CYCLE_OF_L": "CYCLE_OF_LAST_UPDATE",
    "Shape_Leng": "Shape_Length",
    "THICKNESS_": "THICKNESS_FT",
}


# ---------------------------------------------------------------------------
# Rename postdoc shapefile fields to canonical FileGDB names
# ---------------------------------------------------------------------------

postdoc_2015_harmonized = (
    postdoc_2015_saline_10k
    .rename(columns=FIELD_CROSSWALK)
    .copy()
)


# ---------------------------------------------------------------------------
# Compare canonical field sets
# ---------------------------------------------------------------------------

official_fields = set(
    official_saline_10k.columns
)

postdoc_fields = set(
    postdoc_2015_harmonized.columns
)

print(
    "Fields only in official GDB:",
    sorted(official_fields - postdoc_fields),
)

print(
    "\nFields only in postdoc shapefile:",
    sorted(postdoc_fields - official_fields),
)

print(
    "\nSame canonical field set:",
    official_fields == postdoc_fields,
)


# ---------------------------------------------------------------------------
# Show canonical schema side-by-side
# ---------------------------------------------------------------------------

canonical_schema = pd.DataFrame(
    {
        "field": sorted(
            official_fields | postdoc_fields
        )
    }
)

canonical_schema["official_dtype"] = canonical_schema["field"].map(
    {
        column: str(official_saline_10k[column].dtype)
        for column in official_saline_10k.columns
    }
)

canonical_schema["postdoc_dtype"] = canonical_schema["field"].map(
    {
        column: str(postdoc_2015_harmonized[column].dtype)
        for column in postdoc_2015_harmonized.columns
    }
)

canonical_schema["same_dtype"] = (
    canonical_schema["official_dtype"]
    == canonical_schema["postdoc_dtype"]
)

display(canonical_schema)

Fields only in official GDB: []

Fields only in postdoc shapefile: []

Same canonical field set: True


,field,official_dtype,postdoc_dtype,same_dtype
0,ARRA_PROJECT,str,str,True
1,ASSESSED,int16,int32,False
2,BASIN_NAME,str,str,True
3,COL_ROW,str,str,True
4,CYCLE_OF_LAST_UPDATE,str,str,True
5,DEPTH_FT,float64,int64,False
6,DUPLICATE,int16,int32,False
7,MED_CALCED,int16,int32,False
8,OVERLAP,int16,int32,False
9,PARTNERSHIP,str,str,True


Province features: 13
CRS:               EPSG:3347


,PRUID,DGUID,PRNAME,PRENAME,PRFNAME,PREABBR,PRFABBR,LANDAREA,geometry
0,10,2021A000210,Newfoundland and Labrador / Terre-Neuve-et-Lab...,Newfoundland and Labrador,Terre-Neuve-et-Labrador,N.L.,T.-N.-L.,3.581704e+05,"MULTIPOLYGON (((8841194.729 2213093.663, 88411..."
1,11,2021A000211,Prince Edward Island / Île-du-Prince-Édouard,Prince Edward Island,Île-du-Prince-Édouard,P.E.I.,Î.-P.-É.,5.681179e+03,"MULTIPOLYGON (((8374335.443 1629502.597, 83743..."
2,12,2021A000212,Nova Scotia / Nouvelle-Écosse,Nova Scotia,Nouvelle-Écosse,N.S.,N.-É.,5.282471e+04,"MULTIPOLYGON (((8310463.217 1250722.263, 83104..."
3,13,2021A000213,New Brunswick / Nouveau-Brunswick,New Brunswick,Nouveau-Brunswick,N.B.,N.-B.,7.124850e+04,"MULTIPOLYGON (((7964100.72 1576822.289, 796410..."
4,24,2021A000224,Quebec / Québec,Quebec,Québec,Que.,Qc,1.298600e+06,"MULTIPOLYGON (((6948393.211 2760814.626, 69483..."


In [12]:
# ---------------------------------------------------------------------------
# Compare v1502 attribute records independent of row order
# ---------------------------------------------------------------------------

attribute_fields = [
    column
    for column in official_saline_10k.columns
    if column != "geometry"
]

official_attributes = (
    official_saline_10k[attribute_fields]
    .copy()
)

postdoc_attributes = (
    postdoc_2015_harmonized[attribute_fields]
    .copy()
)


# ---------------------------------------------------------------------------
# Normalize values before comparison
# ---------------------------------------------------------------------------

def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize dataframe values so equivalent GDB and shapefile values compare
    consistently despite dtype differences.
    """
    output = df.copy()

    for column in output.columns:

        if pd.api.types.is_numeric_dtype(output[column]):

            output[column] = (
                pd.to_numeric(
                    output[column],
                    errors="coerce",
                )
                .astype("float64")
            )

        else:

            output[column] = (
                output[column]
                .astype("string")
                .str.strip()
            )

    return output


official_normalized = normalize_dataframe(
    official_attributes
)

postdoc_normalized = normalize_dataframe(
    postdoc_attributes
)


# ---------------------------------------------------------------------------
# Build stable row fingerprints
# ---------------------------------------------------------------------------

def row_fingerprints(df: pd.DataFrame) -> pd.Series:
    """
    Generate deterministic hashes from normalized row contents.
    """

    serialized = (
        df
        .fillna("<NULL>")
        .astype(str)
        .agg("|".join, axis=1)
    )

    return serialized.map(
        lambda value: hashlib.sha256(
            value.encode("utf-8")
        ).hexdigest()
    )


official_hashes = row_fingerprints(
    official_normalized
)

postdoc_hashes = row_fingerprints(
    postdoc_normalized
)


# ---------------------------------------------------------------------------
# Compare record multisets
# ---------------------------------------------------------------------------

official_counts = (
    official_hashes
    .value_counts()
    .sort_index()
)

postdoc_counts = (
    postdoc_hashes
    .value_counts()
    .sort_index()
)

all_hashes = (
    official_counts.index
    .union(postdoc_counts.index)
)

record_comparison = pd.DataFrame(
    {
        "official_count": official_counts.reindex(
            all_hashes,
            fill_value=0,
        ),
        "postdoc_count": postdoc_counts.reindex(
            all_hashes,
            fill_value=0,
        ),
    }
)

record_comparison["same_count"] = (
    record_comparison["official_count"]
    == record_comparison["postdoc_count"]
)


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print(
    "Same number of rows:",
    len(official_normalized)
    == len(postdoc_normalized),
)

print(
    "Same attribute record multiset:",
    record_comparison["same_count"].all(),
)

print(
    "Differing record fingerprints:",
    (~record_comparison["same_count"]).sum(),
)

display(
    record_comparison[
        ~record_comparison["same_count"]
    ].head(20)
)

Same number of rows: True
Same attribute record multiset: False
Differing record fingerprints: 372379


,official_count,postdoc_count,same_count
0000132dcf62eeaa6051a257c8b346914b31ad111d2eb090a9b8c4039157ec7e,1,0,False
000014c23c1486d17d40a3edce2a304dea370c2143eabef816d1fb160db8447a,1,0,False
0000245dc2e2aa4fca27710baf8d4ee9e870b58a62cba012a156e1a52ea1aa43,0,1,False
00003878b5a0e033f626f2bc9b7073dad11fa38c78dc2fe1f63f5d8c2d1780da,1,0,False
0000465c2037b8d263391757e3ffbdb6e973a18f4961283b8d66b2911359de88,1,0,False
00008af35ff8bf536d5c1f5ffecc39851d08dcb6322fbe2cc429fefe56778389,0,1,False
0000a6ad03e15979e4f8b731aaa67f2e409135a201da1bbedacf7eb8ec5a706e,0,1,False
00012c6f0199ddee3e37c48e6e1bf2cbcd3a23d6a65163a041f56bdef220e276,1,0,False
00016dfb3ce216831e93ea4e3e92b201fecc606e7514d957154a44e422f99f55,0,1,False
00019681030b967908ee6d6c1439d155a9a43db24bd379ffbf3bff20cdc4e6db,0,1,False


In [15]:
# ---------------------------------------------------------------------------
# Check whether likely identifying fields align by row position
# ---------------------------------------------------------------------------

identity_fields = [
    "COL_ROW",
    "PARTNERSHIP",
    "RESOURCE_NAME",
    "BASIN_NAME",
    "ARRA_PROJECT",
]

print("Row-position agreement for identifying fields:\n")

for column in identity_fields:

    official_values = (
        official_saline_10k[column]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
    )

    postdoc_values = (
        postdoc_2015_harmonized[column]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
    )

    equal = official_values.eq(postdoc_values)

    print(
        f"{column:25} "
        f"{equal.sum():>8,} / {len(equal):,} "
        f"({equal.mean():.6%})"
    )


# ---------------------------------------------------------------------------
# Compare every non-geometry field with dtype-aware logic
# ---------------------------------------------------------------------------

comparison_records = []

for column in attribute_fields:

    official_series = official_saline_10k[column]
    postdoc_series = postdoc_2015_harmonized[column]

    # ---------------------------------------------------------------
    # Numeric fields
    # ---------------------------------------------------------------

    if (
        pd.api.types.is_numeric_dtype(official_series)
        and pd.api.types.is_numeric_dtype(postdoc_series)
    ):

        official_numeric = pd.to_numeric(
            official_series,
            errors="coerce",
        ).astype("float64")

        postdoc_numeric = pd.to_numeric(
            postdoc_series,
            errors="coerce",
        ).astype("float64")

        both_null = (
            official_numeric.isna()
            & postdoc_numeric.isna()
        )

        close = np.isclose(
            official_numeric.fillna(0),
            postdoc_numeric.fillna(0),
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )

        equal_mask = both_null | close

        absolute_difference = (
            official_numeric - postdoc_numeric
        ).abs()

        comparison_records.append(
            {
                "field": column,
                "type": "numeric",
                "rows": len(equal_mask),
                "equal_rows": int(equal_mask.sum()),
                "different_rows": int((~equal_mask).sum()),
                "equal_pct": float(equal_mask.mean() * 100),
                "max_abs_difference": (
                    absolute_difference.max()
                ),
                "mean_abs_difference": (
                    absolute_difference.mean()
                ),
            }
        )

    # ---------------------------------------------------------------
    # Text / categorical fields
    # ---------------------------------------------------------------

    else:

        official_text = (
            official_series
            .astype("string")
            .str.strip()
            .fillna("<NULL>")
        )

        postdoc_text = (
            postdoc_series
            .astype("string")
            .str.strip()
            .fillna("<NULL>")
        )

        equal_mask = official_text.eq(
            postdoc_text
        )

        comparison_records.append(
            {
                "field": column,
                "type": "text",
                "rows": len(equal_mask),
                "equal_rows": int(equal_mask.sum()),
                "different_rows": int((~equal_mask).sum()),
                "equal_pct": float(equal_mask.mean() * 100),
                "max_abs_difference": None,
                "mean_abs_difference": None,
            }
        )


field_value_comparison = pd.DataFrame(
    comparison_records
).sort_values(
    [
        "different_rows",
        "field",
    ],
    ascending=[
        False,
        True,
    ],
)

display(field_value_comparison)

Row-position agreement for identifying fields:

COL_ROW                    186,675 / 186,675 (100.000000%)
PARTNERSHIP                186,675 / 186,675 (100.000000%)
RESOURCE_NAME              186,675 / 186,675 (100.000000%)
BASIN_NAME                 186,675 / 186,675 (100.000000%)
ARRA_PROJECT               186,675 / 186,675 (100.000000%)


,field,type,rows,equal_rows,different_rows,equal_pct,max_abs_difference,mean_abs_difference
14,POROSITY_PCT,numeric,186675,167040,19635,89.48172,0.000050,8.740345e-08
15,PERMEABILITY_mD,numeric,186675,177955,8720,95.32878,0.000500,2.275667e-05
2,ARRA_PROJECT,text,186675,186675,0,100.00000,NaN,NaN
16,ASSESSED,numeric,186675,186675,0,100.00000,0.000000,0.000000e+00
4,BASIN_NAME,text,186675,186675,0,100.00000,NaN,NaN
0,COL_ROW,text,186675,186675,0,100.00000,NaN,NaN
17,CYCLE_OF_LAST_UPDATE,text,186675,186675,0,100.00000,NaN,NaN
9,DEPTH_FT,numeric,186675,186675,0,100.00000,0.000000,0.000000e+00
19,DUPLICATE,numeric,186675,186675,0,100.00000,0.000000,0.000000e+00
20,MED_CALCED,numeric,186675,186675,0,100.00000,0.000000,0.000000e+00


In [16]:
# ---------------------------------------------------------------------------
# Load the postdoc 2022 saline grid and saline polygon layers
# ---------------------------------------------------------------------------

import tempfile
import zipfile

import geopandas as gpd
from pathlib import Path


def load_single_shapefile_from_zip(zip_path: Path) -> gpd.GeoDataFrame:
    """Extract a ZIP temporarily and load its single shapefile."""

    with tempfile.TemporaryDirectory() as temp_dir:

        temp_dir = Path(temp_dir)

        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(temp_dir)

        shapefiles = sorted(
            temp_dir.rglob("*.shp")
        )

        if len(shapefiles) != 1:
            raise ValueError(
                f"Expected exactly one shapefile in {zip_path.name}, "
                f"found {len(shapefiles)}: "
                f"{[path.name for path in shapefiles]}"
            )

        return gpd.read_file(
            shapefiles[0]
        )


# ---------------------------------------------------------------------------
# Load 2022 grid
# ---------------------------------------------------------------------------

postdoc_2022_saline_10k = load_single_shapefile_from_zip(
    POSTDOC_2022_GRID_ZIP
)


# ---------------------------------------------------------------------------
# Load accompanying saline polygon layer
# ---------------------------------------------------------------------------

postdoc_saline_poly = load_single_shapefile_from_zip(
    POSTDOC_SALINE_POLY_ZIP
)


# ---------------------------------------------------------------------------
# Summarize
# ---------------------------------------------------------------------------

print("Postdoc 2022 saline 10 km grid")
print("--------------------------------")
print(f"Rows:          {len(postdoc_2022_saline_10k):,}")
print(f"Columns:       {len(postdoc_2022_saline_10k.columns):,}")
print(f"CRS:           {postdoc_2022_saline_10k.crs}")
print(
    "Geometry type:",
    postdoc_2022_saline_10k
    .geom_type
    .value_counts()
    .to_dict(),
)


print("\nPostdoc saline polygon layer")
print("------------------------------")
print(f"Rows:          {len(postdoc_saline_poly):,}")
print(f"Columns:       {len(postdoc_saline_poly.columns):,}")
print(f"CRS:           {postdoc_saline_poly.crs}")
print(
    "Geometry type:",
    postdoc_saline_poly
    .geom_type
    .value_counts()
    .to_dict(),
)


# ---------------------------------------------------------------------------
# Compare broad structure with v1502
# ---------------------------------------------------------------------------

structure_2022 = pd.DataFrame(
    [
        {
            "dataset": "Official NATCARB v1502 saline 10 km",
            "rows": len(official_saline_10k),
            "columns": len(official_saline_10k.columns),
            "crs": str(official_saline_10k.crs),
        },
        {
            "dataset": "Postdoc 2022 saline 10 km",
            "rows": len(postdoc_2022_saline_10k),
            "columns": len(postdoc_2022_saline_10k.columns),
            "crs": str(postdoc_2022_saline_10k.crs),
        },
        {
            "dataset": "Postdoc saline polygons",
            "rows": len(postdoc_saline_poly),
            "columns": len(postdoc_saline_poly.columns),
            "crs": str(postdoc_saline_poly.crs),
        },
    ]
)

display(structure_2022)

Postdoc 2022 saline 10 km grid
--------------------------------
Rows:          186,675
Columns:       54
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",45],PARAMETER["longitude_of_center",-100],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry type: {'Polygon': 186675}

Postdoc saline polygon layer
------------------------------
Rows:          508
Columns:       11
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_E

,dataset,rows,columns,crs
0,Official NATCARB v1502 saline 10 km,186675,24,"PROJCS[""Lambert Azimuthal Equal-area"",GEOGCS[""..."
1,Postdoc 2022 saline 10 km,186675,54,"PROJCS[""Lambert Azimuthal Equal-area"",GEOGCS[""..."
2,Postdoc saline polygons,508,11,"PROJCS[""Lambert Azimuthal Equal-area"",GEOGCS[""..."


In [17]:
# ---------------------------------------------------------------------------
# Compare v1502 and 2022 schemas
# ---------------------------------------------------------------------------

# Harmonize the original shapefile-style field names in the 2022 layer
# to the canonical FileGDB names used in the official v1502 dataset.

postdoc_2022_harmonized = (
    postdoc_2022_saline_10k
    .rename(columns=FIELD_CROSSWALK)
    .copy()
)


# ---------------------------------------------------------------------------
# Compare field sets
# ---------------------------------------------------------------------------

v1502_fields = set(
    official_saline_10k.columns
)

fields_2022 = set(
    postdoc_2022_harmonized.columns
)

shared_fields = sorted(
    v1502_fields & fields_2022
)

new_2022_fields = sorted(
    fields_2022 - v1502_fields
)

missing_2022_fields = sorted(
    v1502_fields - fields_2022
)


print(f"Shared fields:        {len(shared_fields)}")
print(f"New 2022 fields:      {len(new_2022_fields)}")
print(f"Missing v1502 fields: {len(missing_2022_fields)}")


print("\nNew fields introduced in 2022:")
for field in new_2022_fields:
    print(f"  - {field}")


print("\nOriginal fields missing from 2022:")
if missing_2022_fields:
    for field in missing_2022_fields:
        print(f"  - {field}")
else:
    print("  None")


# ---------------------------------------------------------------------------
# Inspect new identifiers
# ---------------------------------------------------------------------------

identifier_fields = [
    field
    for field in ["UID", "New_ID"]
    if field in postdoc_2022_harmonized.columns
]

identifier_summary = []

for field in identifier_fields:

    series = postdoc_2022_harmonized[field]

    identifier_summary.append(
        {
            "field": field,
            "rows": len(series),
            "non_null": int(series.notna().sum()),
            "null": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True)),
            "duplicate_rows": int(
                series.duplicated(keep=False).sum()
            ),
            "is_unique": bool(
                series.dropna().is_unique
            ),
        }
    )

identifier_summary = pd.DataFrame(
    identifier_summary
)

display(identifier_summary)


# ---------------------------------------------------------------------------
# Inspect example identifier relationships
# ---------------------------------------------------------------------------

preview_fields = [
    field
    for field in [
        "New_ID",
        "UID",
        "COL_ROW",
        "RESOURCE_NAME",
        "BASIN_NAME",
    ]
    if field in postdoc_2022_harmonized.columns
]

display(
    postdoc_2022_harmonized[
        preview_fields
    ].head(20)
)

Shared fields:        21
New 2022 fields:      33
Missing v1502 fields: 3

New fields introduced in 2022:
  - Avg_Porosi
  - Avg_depth_
  - Bottom__de
  - Bottom_dep
  - Citation_4
  - Citation_5
  - Citation_6
  - Deposition
  - Formation_
  - Lithology
  - Max_Porosi
  - Maximum_Pe
  - Min_Porosi
  - Minimum_Pe
  - Minor_Lith
  - Minor_depo
  - New_ID
  - Notes_on_T
  - Permeabi_1
  - Shape_Le_1
  - Source3_1
  - Source_12
  - Source_23
  - Source_f_1
  - Source_for
  - Thicknes_1
  - Thicknes_2
  - Thickness1
  - Top_depth1
  - Top_depth_
  - UID
  - USGS_sau_1
  - USGS_sau_B

Original fields missing from 2022:
  - DUPLICATE
  - MED_CALCED
  - Shape_Length


,field,rows,non_null,null,unique_values,duplicate_rows,is_unique
0,UID,186675,186675,0,505,186673,False
1,New_ID,186675,186675,0,186675,0,True


,New_ID,UID,COL_ROW,RESOURCE_NAME,BASIN_NAME
0,1,304,314 - 367,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
1,2,304,314 - 366,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
2,3,304,314 - 365,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
3,4,304,314 - 364,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
4,5,304,315 - 367,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
5,6,304,315 - 366,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
6,7,304,315 - 365,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
7,8,304,316 - 368,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
8,9,304,316 - 367,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."
9,10,304,316 - 366,Basal Cambrian,"Alberta Basin, Williston Basin, Central Montan..."


In [18]:
# ---------------------------------------------------------------------------
# Inspect polygon schema and UID linkage to the 2022 10 km grid
# ---------------------------------------------------------------------------

print("Postdoc saline polygon fields:\n")

for column in postdoc_saline_poly.columns:
    print(f"  - {column}")


# ---------------------------------------------------------------------------
# Identify likely UID fields in polygon layer
# ---------------------------------------------------------------------------

uid_candidates = [
    column
    for column in postdoc_saline_poly.columns
    if "uid" in column.lower()
    or "id" == column.lower()
    or column.lower().endswith("_id")
]

print("\nCandidate polygon identifier fields:")
for column in uid_candidates:
    print(f"  - {column}")


# ---------------------------------------------------------------------------
# Summarize candidate identifiers
# ---------------------------------------------------------------------------

polygon_id_summary = []

for column in uid_candidates:

    series = postdoc_saline_poly[column]

    polygon_id_summary.append(
        {
            "field": column,
            "rows": len(series),
            "non_null": int(series.notna().sum()),
            "null": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True)),
            "is_unique": bool(series.dropna().is_unique),
        }
    )

display(
    pd.DataFrame(polygon_id_summary)
)

Postdoc saline polygon fields:

  - PARTNERSHI
  - ARRA_PROJE
  - RESOURCE_N
  - BASIN_NAME
  - ASSESSED
  - CYCLE_OF_L
  - OVERLAP
  - DUPLICATE
  - Shape_Leng
  - Shape_Area
  - geometry

Candidate polygon identifier fields:


""


In [19]:
# ---------------------------------------------------------------------------
# Test whether 2022 UID corresponds to formation/resource + basin identity
# ---------------------------------------------------------------------------

grid_uid_groups = (
    postdoc_2022_harmonized
    .groupby("UID", dropna=False)
    .agg(
        resource_names=("RESOURCE_NAME", lambda x: sorted(set(x.dropna().astype(str)))),
        basin_names=("BASIN_NAME", lambda x: sorted(set(x.dropna().astype(str)))),
        grid_cells=("New_ID", "count"),
    )
    .reset_index()
)

grid_uid_groups["n_resource_names"] = (
    grid_uid_groups["resource_names"]
    .map(len)
)

grid_uid_groups["n_basin_names"] = (
    grid_uid_groups["basin_names"]
    .map(len)
)


# ---------------------------------------------------------------------------
# Check whether each UID maps to exactly one resource/basin combination
# ---------------------------------------------------------------------------

print(
    "UIDs with exactly one RESOURCE_NAME:",
    (grid_uid_groups["n_resource_names"] == 1).sum(),
    "/",
    len(grid_uid_groups),
)

print(
    "UIDs with exactly one BASIN_NAME:",
    (grid_uid_groups["n_basin_names"] == 1).sum(),
    "/",
    len(grid_uid_groups),
)

print(
    "UIDs with one resource and one basin:",
    (
        (grid_uid_groups["n_resource_names"] == 1)
        & (grid_uid_groups["n_basin_names"] == 1)
    ).sum(),
    "/",
    len(grid_uid_groups),
)


# ---------------------------------------------------------------------------
# Inspect non-unique UID relationships
# ---------------------------------------------------------------------------

ambiguous_uids = grid_uid_groups[
    (grid_uid_groups["n_resource_names"] != 1)
    | (grid_uid_groups["n_basin_names"] != 1)
].copy()

display(ambiguous_uids.head(30))

UIDs with exactly one RESOURCE_NAME: 502 / 505
UIDs with exactly one BASIN_NAME: 479 / 505
UIDs with one resource and one basin: 476 / 505


,UID,resource_names,basin_names,grid_cells,n_resource_names,n_basin_names
12,13,[Astoria-Nehalem],[],64,1,0
18,19,[Big Valley],[],9,1,0
39,40,[Coos],[],40,1,0
57,58,[Harney],[],132,1,0
64,65,[Hornbrook],[],17,1,0
85,86,[Methow],[],46,1,0
86,87,[Methow Trough],[],38,1,0
94,95,[Nevada],[],86,1,0
100,101,[Ochoco],[],273,1,0
115,116,[Puget Sound],[],303,1,0


In [20]:
# ---------------------------------------------------------------------------
# Compare formation/basin combinations between grid and polygon layers
# ---------------------------------------------------------------------------

polygon_harmonized = (
    postdoc_saline_poly
    .rename(columns=FIELD_CROSSWALK)
    .copy()
)


grid_pairs = (
    postdoc_2022_harmonized[
        [
            "UID",
            "RESOURCE_NAME",
            "BASIN_NAME",
        ]
    ]
    .drop_duplicates()
)

polygon_pairs = (
    polygon_harmonized[
        [
            "RESOURCE_NAME",
            "BASIN_NAME",
        ]
    ]
    .drop_duplicates()
)


print(f"Distinct UID/resource/basin rows in grid: {len(grid_pairs):,}")
print(f"Distinct resource/basin pairs in polygons: {len(polygon_pairs):,}")


pair_comparison = (
    grid_pairs
    .merge(
        polygon_pairs,
        on=[
            "RESOURCE_NAME",
            "BASIN_NAME",
        ],
        how="outer",
        indicator=True,
    )
)

print("\nPair match status:")
display(
    pair_comparison["_merge"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)

Distinct UID/resource/basin rows in grid: 508
Distinct resource/basin pairs in polygons: 498

Pair match status:


,status,rows
0,left_only,293
1,right_only,288
2,both,215


In [21]:
# ---------------------------------------------------------------------------
# Compare postdoc saline polygon ZIP with official NATCARB v1502 polygon layer
# ---------------------------------------------------------------------------

OFFICIAL_SALINE_POLY_LAYER = "NATCARB_Saline_Poly_v1502"

official_saline_poly = gpd.read_file(
    OFFICIAL_GDB,
    layer=OFFICIAL_SALINE_POLY_LAYER,
)


# ---------------------------------------------------------------------------
# Harmonize shapefile field names
# ---------------------------------------------------------------------------

postdoc_poly_harmonized = (
    postdoc_saline_poly
    .rename(columns=FIELD_CROSSWALK)
    .copy()
)


# ---------------------------------------------------------------------------
# Structural comparison
# ---------------------------------------------------------------------------

print("Official v1502 saline polygon layer")
print("------------------------------------")
print(f"Rows:          {len(official_saline_poly):,}")
print(f"Columns:       {len(official_saline_poly.columns):,}")
print(f"CRS:           {official_saline_poly.crs}")
print(
    "Geometry type:",
    official_saline_poly.geom_type.value_counts().to_dict(),
)

print("\nPostdoc saline polygon ZIP")
print("---------------------------")
print(f"Rows:          {len(postdoc_poly_harmonized):,}")
print(f"Columns:       {len(postdoc_poly_harmonized.columns):,}")
print(f"CRS:           {postdoc_poly_harmonized.crs}")
print(
    "Geometry type:",
    postdoc_poly_harmonized.geom_type.value_counts().to_dict(),
)


# ---------------------------------------------------------------------------
# Compare logical schemas
# ---------------------------------------------------------------------------

official_poly_fields = set(
    official_saline_poly.columns
)

postdoc_poly_fields = set(
    postdoc_poly_harmonized.columns
)

print(
    "\nFields only in official:",
    sorted(official_poly_fields - postdoc_poly_fields),
)

print(
    "Fields only in postdoc:",
    sorted(postdoc_poly_fields - official_poly_fields),
)

print(
    "Same logical field set:",
    official_poly_fields == postdoc_poly_fields,
)


# ---------------------------------------------------------------------------
# Check row-position agreement for key descriptive fields
# ---------------------------------------------------------------------------

poly_identity_fields = [
    field
    for field in [
        "PARTNERSHIP",
        "ARRA_PROJECT",
        "RESOURCE_NAME",
        "BASIN_NAME",
        "ASSESSED",
        "CYCLE_OF_LAST_UPDATE",
        "OVERLAP",
        "DUPLICATE",
    ]
    if (
        field in official_saline_poly.columns
        and field in postdoc_poly_harmonized.columns
    )
]

print("\nRow-position agreement:\n")

for field in poly_identity_fields:

    official_values = (
        official_saline_poly[field]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
    )

    postdoc_values = (
        postdoc_poly_harmonized[field]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
    )

    equal = official_values.eq(postdoc_values)

    print(
        f"{field:25} "
        f"{equal.sum():>5,} / {len(equal):,} "
        f"({equal.mean():.6%})"
    )

Official v1502 saline polygon layer
------------------------------------
Rows:          508
Columns:       11
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",45],PARAMETER["longitude_of_center",-100],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Geometry type: {'MultiPolygon': 508}

Postdoc saline polygon ZIP
---------------------------
Rows:          508
Columns:       11
CRS:           PROJCS["Lambert Azimuthal Equal-area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal

c:\Users\aviga\Research\repos\canco2-storage\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


In [22]:
# ---------------------------------------------------------------------------
# Quantify coverage of all fields introduced in the 2022 enrichment
# ---------------------------------------------------------------------------

new_field_coverage = []

for field in new_2022_fields:

    series = postdoc_2022_harmonized[field]

    non_null = int(series.notna().sum())
    unique = int(series.nunique(dropna=True))

    new_field_coverage.append(
        {
            "field": field,
            "non_null_rows": non_null,
            "null_rows": len(series) - non_null,
            "coverage_pct": 100 * non_null / len(series),
            "unique_values": unique,
        }
    )

new_field_coverage = (
    pd.DataFrame(new_field_coverage)
    .sort_values(
        ["coverage_pct", "field"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(new_field_coverage)

,field,non_null_rows,null_rows,coverage_pct,unique_values
0,Avg_depth_,186675,0,100.000000,25
1,New_ID,186675,0,100.000000,186675
2,Shape_Le_1,186675,0,100.000000,1
3,Top_depth_,186675,0,100.000000,14
4,UID,186675,0,100.000000,505
5,Formation_,164955,21720,88.364805,209
6,Lithology,148980,37695,79.807151,8
7,Deposition,145795,40880,78.100978,24
8,Citation_4,138328,48347,74.100978,158
9,Source_12,138093,48582,73.975090,166


In [23]:
# ---------------------------------------------------------------------------
# Measure enrichment coverage at the formation/entity UID level
# ---------------------------------------------------------------------------

uid_field_coverage = []

for field in new_2022_fields:

    if field in {"UID", "New_ID"}:
        continue

    coverage_by_uid = (
        postdoc_2022_harmonized
        .groupby("UID")[field]
        .apply(lambda x: x.notna().any())
    )

    uid_field_coverage.append(
        {
            "field": field,
            "uids_with_data": int(coverage_by_uid.sum()),
            "total_uids": len(coverage_by_uid),
            "uid_coverage_pct": 100 * coverage_by_uid.mean(),
        }
    )

uid_field_coverage = (
    pd.DataFrame(uid_field_coverage)
    .sort_values(
        ["uid_coverage_pct", "field"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(uid_field_coverage)

,field,uids_with_data,total_uids,uid_coverage_pct
0,Avg_depth_,505,505,100.000000
1,Shape_Le_1,505,505,100.000000
2,Top_depth_,505,505,100.000000
3,Lithology,368,505,72.871287
4,Deposition,363,505,71.881188
5,Source_12,357,505,70.693069
6,Citation_4,349,505,69.108911
7,Formation_,338,505,66.930693
8,Minor_Lith,237,505,46.930693
9,Minor_depo,236,505,46.732673


In [24]:
# ---------------------------------------------------------------------------
# Inspect suspicious fully populated 2022 fields
# ---------------------------------------------------------------------------

fields_to_inspect = [
    "Avg_depth_",
    "Top_depth_",
    "Shape_Le_1",
]

for field in fields_to_inspect:

    print(f"\n{field}")
    print("-" * len(field))

    display(
        postdoc_2022_harmonized[field]
        .value_counts(dropna=False)
        .rename_axis(field)
        .reset_index(name="rows")
        .head(30)
    )


Avg_depth_
----------


,Avg_depth_,rows
0,0.000,173517
1,3962.789,2477
2,10250.000,1685
3,9797.000,1592
4,5400.355,1098
5,10000.000,931
6,9000.000,846
7,6020.000,802
8,10430.000,707
9,9451.000,664



Top_depth_
----------


,Top_depth_,rows
0,0.0,165224
1,3280.0,5678
2,3441.6,2477
3,9000.0,2364
4,4921.0,2278
5,3000.0,1970
6,8200.0,1685
7,3450.0,1350
8,1941.0,1098
9,6020.0,802



Shape_Le_1
----------


,Shape_Le_1,rows
0,40000.0,186675


In [25]:
# ---------------------------------------------------------------------------
# Recalculate effective coverage for 2022 numeric enrichment fields
# treating zero-valued placeholders as missing where appropriate
# ---------------------------------------------------------------------------

zero_as_missing_fields = [
    "Avg_depth_",
    "Top_depth_",
    "Bottom__de",
    "Bottom_dep",
    "Avg_Porosi",
    "Min_Porosi",
    "Max_Porosi",
    "Permeabi_1",
    "Minimum_Pe",
    "Maximum_Pe",
    "Thickness1",
    "Thicknes_1",
    "Thicknes_2",
]

effective_coverage = []

for field in zero_as_missing_fields:

    if field not in postdoc_2022_harmonized.columns:
        continue

    series = pd.to_numeric(
        postdoc_2022_harmonized[field],
        errors="coerce",
    )

    valid = series.notna() & (series != 0)

    effective_coverage.append(
        {
            "field": field,
            "valid_rows": int(valid.sum()),
            "total_rows": len(series),
            "effective_coverage_pct": 100 * valid.mean(),
            "nonzero_unique_values": int(
                series[valid].nunique()
            ),
        }
    )

effective_coverage = (
    pd.DataFrame(effective_coverage)
    .sort_values(
        "effective_coverage_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(effective_coverage)

,field,valid_rows,total_rows,effective_coverage_pct,nonzero_unique_values
0,Avg_Porosi,35869,186675,19.214678,16
1,Max_Porosi,35444,186675,18.987010,18
2,Min_Porosi,31288,186675,16.760680,15
3,Permeabi_1,30238,186675,16.198205,16
4,Minimum_Pe,24832,186675,13.302263,11
5,Maximum_Pe,24832,186675,13.302263,16
6,Top_depth_,21451,186675,11.491094,13
7,Thicknes_2,15299,186675,8.195527,27
8,Avg_depth_,13158,186675,7.048614,24
9,Thicknes_1,10082,186675,5.400830,10


In [26]:
# ---------------------------------------------------------------------------
# Effective quantitative-property coverage at the UID level
# ---------------------------------------------------------------------------

uid_effective_coverage = []

for field in zero_as_missing_fields:

    if field not in postdoc_2022_harmonized.columns:
        continue

    values = pd.to_numeric(
        postdoc_2022_harmonized[field],
        errors="coerce",
    )

    valid = values.notna() & (values != 0)

    temp = pd.DataFrame(
        {
            "UID": postdoc_2022_harmonized["UID"],
            "valid": valid,
        }
    )

    uid_has_data = (
        temp
        .groupby("UID")["valid"]
        .any()
    )

    uid_effective_coverage.append(
        {
            "field": field,
            "uids_with_data": int(uid_has_data.sum()),
            "total_uids": len(uid_has_data),
            "uid_coverage_pct": 100 * uid_has_data.mean(),
        }
    )

uid_effective_coverage = (
    pd.DataFrame(uid_effective_coverage)
    .sort_values(
        "uid_coverage_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(uid_effective_coverage)

,field,uids_with_data,total_uids,uid_coverage_pct
0,Max_Porosi,47,505,9.306931
1,Avg_Porosi,38,505,7.524752
2,Min_Porosi,37,505,7.326733
3,Permeabi_1,36,505,7.128713
4,Avg_depth_,30,505,5.940594
5,Maximum_Pe,30,505,5.940594
6,Thicknes_2,30,505,5.940594
7,Minimum_Pe,30,505,5.940594
8,Top_depth_,25,505,4.950495
9,Thicknes_1,11,505,2.178218


## Conclusions

This notebook compared the original NATCARB v1502 saline storage datasets with a postdoc-prepared 2022 NATCARB-derived dataset to determine whether the 2022 material should replace, modify, or remain separate from the original NATCARB source.

### Baseline comparison

The postdoc copies of the original NATCARB saline datasets were found to be effectively equivalent to the official NATCARB v1502 File Geodatabase layers.

For the saline 10 km grid:

- official NATCARB v1502 contains 186,675 records,
- the postdoc `saline10k_1502` shapefile also contains 186,675 records,
- both use the same Lambert Azimuthal Equal Area CRS,
- their logical schemas are identical after accounting for the 10-character shapefile field-name limit,
- identifying fields match row-for-row,
- substantive numeric fields match within expected serialization precision,
- observed differences in data types and polygon representation are consistent with FileGDB-to-shapefile export.

For the saline polygon layer:

- both the official and postdoc versions contain 508 records,
- both contain the same 11 logical fields,
- identifying and descriptive attributes match row-for-row,
- geometry representation differs only in FileGDB versus shapefile serialization.

The postdoc copies of the 2015 datasets can therefore be treated as alternate file-format representations of the official NATCARB v1502 release rather than independent data sources.

### 2022 derived dataset

The 2022 saline 10 km dataset retains the same 186,675 grid records but expands the schema from 24 to 54 columns.

The 2022 workflow introduces:

- `New_ID`, a unique identifier for each grid record,
- `UID`, a repeated identifier grouping grid cells into approximately 505 formation-level entities,
- additional formation names,
- lithology and depositional environment,
- minor lithology and depositional descriptors,
- porosity and permeability attributes,
- additional depth and thickness attributes,
- USGS storage-unit crosswalks,
- source and citation information.

This is consistent with the accompanying documentation, which describes the 2022 work as adding formation-level information to the existing NATCARB saline grid rather than replacing the original grid. The documentation also references an intermediate `Saline_Poly_Altas5_v2` layer used in the enrichment workflow. That layer was not present in the supplied files. :contentReference[oaicite:0]{index=0}

The supplied saline polygon ZIP is simply the original NATCARB v1502 polygon dataset and does not contain the 2022 `UID` crosswalk. As a result, the complete formation-level lineage of the 2022 enrichment cannot currently be reconstructed from the available polygon file.

### Coverage of 2022 enrichment

The 2022 enrichment has relatively broad coverage for descriptive geological information but limited coverage for quantitative reservoir properties.

At the `UID` level:

- lithology is available for approximately 73% of UIDs,
- depositional environment for approximately 72%,
- formation-name enrichment for approximately 67%,
- maximum porosity for approximately 9%,
- average porosity for approximately 8%,
- permeability for approximately 7%,
- average depth for approximately 6%,
- minimum and maximum permeability for approximately 6%,
- most additional thickness and bottom-depth fields for substantially less than 6%.

Several fields initially appeared fully populated because zero values were used as placeholders. For example, most records in the added depth fields contain `0`, so their effective coverage is much lower than their raw non-null coverage.

`Shape_Le_1` is equal to 40,000 for every grid record and appears to represent the perimeter of the nominal 10 km grid cell rather than a reservoir property.

### Interpretation

The 2022 dataset should not be treated as a complete replacement for NATCARB v1502.

Instead, it is best interpreted as a **derived companion or enrichment source** built on the NATCARB v1502 spatial resource base.

The following provenance relationship is recommended:

```text
NATCARB v1502
    canonical source dataset
        |
        v
NATCARB 2022 enrichment
    derived / supplementary source